In [2]:
import torch
import torch.nn as nn
from model_def import DistillTriClassModel

PTH_PATH = r"D:\2D_online_system\Blacktea_Withering_System\基于近红外和机器视觉的预训练自监督学习多模态三分类模型\best_model.pth"
ONNX_PATH = "wither_3class_multimodal.onnx"

def smart_load(model: nn.Module, pth_path: str) -> nn.Module:
    obj = torch.load(pth_path, map_location="cpu")

    # 1) checkpoint dict: {"model_state_dict": ...}
    if isinstance(obj, dict) and "model_state_dict" in obj:
        model.load_state_dict(obj["model_state_dict"], strict=True)
        return model

    # 2) pure state_dict (most common)
    if isinstance(obj, dict) and all(isinstance(k, str) for k in obj.keys()):
        model.load_state_dict(obj, strict=True)
        return model

    # 3) full model saved by torch.save(model)
    if isinstance(obj, nn.Module):
        return obj

    raise RuntimeError(f"Unrecognized pth format: {type(obj)}")

class ExportWrap(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, image, nir):
        out = self.m(image, nir)
        if isinstance(out, dict):
            return out.get("logits", next(iter(out.values())))
        if isinstance(out, (tuple, list)):
            return out[0]
        return out

def main():
    print("Loading model...")
    model = DistillTriClassModel(num_classes=3)
    model = smart_load(model, PTH_PATH)
    print("Model loaded successfully.")

    model.eval()
    model.to("cpu")

    # 你的模型固定输入：image(1,3,224,224) + nir(1,128)
    img = torch.randn(1, 3, 224, 224, dtype=torch.float32)
    nir = torch.randn(1, 128, dtype=torch.float32)

    # smoke test
    with torch.no_grad():
        y = model(img, nir)
    if isinstance(y, dict):
        y = y.get("logits", next(iter(y.values())))
    elif isinstance(y, (tuple, list)):
        y = y[0]
    print("Smoke test output shape:", tuple(y.shape))  # 期望 (1,3)

    print("Wrapping the model for ONNX export...")
    wrap = ExportWrap(model)

    print("Starting ONNX export...")
    torch.onnx.export(
        wrap,
        (img, nir),
        ONNX_PATH,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["image", "nir"],
        output_names=["logits"],
        dynamic_axes={
            "image": {0: "batch"},
            "nir": {0: "batch"},
            "logits": {0: "batch"},
        },
    )

    print("Exported ONNX to:", ONNX_PATH)

if __name__ == "__main__":
    main()


OSError: [WinError 127] 找不到指定的程序。 Error loading "c:\Users\79365\anaconda3\envs\42\Lib\site-packages\torch\lib\c10_cuda.dll" or one of its dependencies.